# **1. Data collection**

This notebook collects the official climate, disaster-impact, and agricultural-yield datasets selected for the Pacific Dataviz Challenge 2026. Each source is downloaded from the SPC Pacific Data Hub .Stat API and recorded with its URL, retrieval time, file size, and row count.

Raw files are stored unchanged so the analysis remains auditable and reproducible.

In [ ]:
from __future__ import annotations

import csv
import json
import shutil
import subprocess
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import quote
from urllib.request import Request, urlopen

import pandas as pd
from IPython.display import Markdown, display

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_colwidth', 160)


def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / 'requirements.txt').exists() or (candidate / '.git').exists():
            return candidate
    return cwd


PROJECT_ROOT = find_project_root()
RAW_DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
DOTSTAT_API = 'https://stats-nsi-stable.pacificdata.org/rest/data'
USER_AGENT = 'pacific-dataviz-challenge-26-data-collection/1.0'

FORCE_DOWNLOAD = False
CLEAN_UNSELECTED_DATASETS = True

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'RAW_DATA_DIR: {RAW_DATA_DIR}')

## **1.1. Selected datasets**

The catalog covers eight indicators across climate conditions, disaster impacts, and food systems. Dataset identifiers and query keys are kept beside their human-readable titles so every download can be traced to the official source.

In [ ]:
SELECTED_DOTSTAT_DATASETS = [
    {
        'slug': 'mean_surface_temperature_anomalies',
        'title': 'Mean surface temperature anomalies',
        'front': 'land_temperature',
        'agency': 'SPC',
        'dataflow': 'DF_CLIMATE_CHANGE',
        'version': '1.0',
        'key': 'A.ST_ANOM.',
        'source_url': 'https://stats.pacificdata.org/vis?lc=en&df[ds]=SPC2&df[id]=DF_CLIMATE_CHANGE&df[ag]=SPC&df[vs]=1.0&dq=A.ST_ANOM.&pd=,&to[TIME_PERIOD]=false',
    },
    {
        'slug': 'mean_sea_surface_temperature_anomalies',
        'title': 'Mean sea surface temperature anomalies',
        'front': 'sea_temperature',
        'agency': 'SPC',
        'dataflow': 'DF_CLIMATE_CHANGE',
        'version': '1.0',
        'key': 'A.SST_ANOM.',
        'source_url': 'https://stats.pacificdata.org/vis?lc=en&df[ds]=SPC2&df[id]=DF_CLIMATE_CHANGE&df[ag]=SPC&df[vs]=1.0&dq=A.SST_ANOM.&pd=,&to[TIME_PERIOD]=false',
    },
    {
        'slug': 'rainfall_anomalies',
        'title': 'Rainfall anomalies',
        'front': 'rainfall',
        'agency': 'SPC',
        'dataflow': 'DF_CLIMATE_CHANGE',
        'version': '1.0',
        'key': 'A.RAIN_ANOM.',
        'source_url': 'https://stats.pacificdata.org/vis?lc=en&df[ds]=SPC2&df[id]=DF_CLIMATE_CHANGE&df[ag]=SPC&df[vs]=1.0&dq=A.RAIN_ANOM.&pd=,&to[TIME_PERIOD]=false',
    },
    {
        'slug': 'disaster_directly_affected_persons',
        'title': 'Number of directly affected persons attributed to disasters',
        'front': 'disaster_people',
        'agency': 'SPC',
        'dataflow': 'DF_SDG_11',
        'version': '3.0',
        'key': 'A.VC_DSR_AFFCT.........',
        'source_url': 'https://stats.pacificdata.org/vis?lc=en&df[ds]=ds%3ASPC2&df[id]=DF_SDG_11&df[ag]=SPC&df[vs]=3.0&dq=A.VC_DSR_AFFCT.........&pd=,&to[TIME_PERIOD]=false&lb=bt',
    },
    {
        'slug': 'direct_disaster_economic_loss',
        'title': 'Direct disaster economic loss',
        'front': 'disaster_economic_loss',
        'agency': 'SPC',
        'dataflow': 'DF_SDG_11',
        'version': '3.0',
        'key': 'A.VC_DSR_AALT...._T.....',
        'source_url': 'https://stats.pacificdata.org/vis?lc=en&df[ds]=ds%3ASPC2&df[id]=DF_SDG_11&df[ag]=SPC&df[vs]=3.0&dq=A.VC_DSR_AALT...._T.....&pd=,&to[TIME_PERIOD]=false',
    },
    {
        'slug': 'crop_yield',
        'title': 'Crop yield',
        'front': 'food_systems',
        'agency': 'SPC',
        'dataflow': 'DF_CLIMATE_CHANGE',
        'version': '1.0',
        'key': 'A.CROP_YIELD.',
        'source_url': 'https://stats.pacificdata.org/vis?lc=en&df[ds]=SPC2&df[id]=DF_CLIMATE_CHANGE&df[ag]=SPC&df[vs]=1.0&dq=A.CROP_YIELD.&pd=,&to[TIME_PERIOD]=false',
    },
    {
        'slug': 'livestock_yield',
        'title': 'Livestock yield',
        'front': 'food_systems',
        'agency': 'SPC',
        'dataflow': 'DF_CLIMATE_CHANGE',
        'version': '1.0',
        'key': 'A.LVST_YIELD.',
        'source_url': 'https://stats.pacificdata.org/vis?lc=en&df[ds]=SPC2&df[id]=DF_CLIMATE_CHANGE&df[ag]=SPC&df[vs]=1.0&dq=A.LVST_YIELD.&pd=,&to[TIME_PERIOD]=false',
    },
    {
        'slug': 'agricultural_yield_disaggregated',
        'title': 'Agricultural yield - disaggregated',
        'front': 'food_systems',
        'agency': 'SPC',
        'dataflow': 'DF_AGRICULTURAL_PRODUCTION',
        'version': '1.0',
        'key': 'A...',
        'source_url': 'https://stats.pacificdata.org/vis?lc=en&df[ds]=SPC2&df[id]=DF_AGRICULTURAL_PRODUCTION&df[ag]=SPC&df[vs]=1.0&dq=A...&pd=,&to[TIME_PERIOD]=false',
    },
]


def dotstat_url(dataset: dict[str, str]) -> str:
    flow = ','.join([dataset['agency'], dataset['dataflow'], dataset['version']])
    key = quote(dataset['key'], safe='.+_')
    return f'{DOTSTAT_API}/{flow}/{key}?dimensionAtObservation=AllDimensions&format=csv'


catalog = pd.DataFrame(SELECTED_DOTSTAT_DATASETS)
catalog['api_url'] = catalog.apply(lambda row: dotstat_url(row.to_dict()), axis=1)
display(catalog[['slug', 'front', 'title', 'dataflow', 'key']])

## **1.2. Download helpers**

Downloads are resumable by default: existing non-empty files are retained unless `FORCE_DOWNLOAD` is enabled. The helpers also write deterministic metadata and count source rows for the collection manifest.

In [ ]:
def run_curl(url: str, target: Path) -> None:
    target.parent.mkdir(parents=True, exist_ok=True)
    cmd = [
        'curl', '-L', '--fail', '--silent', '--show-error',
        '--retry', '4', '--retry-delay', '2',
        '--user-agent', USER_AGENT,
        '--output', str(target), url,
    ]
    subprocess.run(cmd, check=True)


def download_text(url: str, target: Path) -> None:
    target.parent.mkdir(parents=True, exist_ok=True)
    request = Request(url, headers={'User-Agent': USER_AGENT, 'Accept-Language': 'en'})
    with urlopen(request, timeout=120) as response:
        target.write_bytes(response.read())


def download_file(url: str, target: Path, *, force: bool = False) -> None:
    if target.exists() and target.stat().st_size > 0 and not force:
        print(f'skip existing {target.relative_to(PROJECT_ROOT)}')
        return
    print(f'download {target.relative_to(PROJECT_ROOT)}')
    if shutil.which('curl'):
        run_curl(url, target)
    else:
        download_text(url, target)


def write_json(path: Path, payload: dict) -> None:
    path.write_text(json.dumps(payload, indent=2, sort_keys=True) + '\n', encoding='utf-8')


def count_csv_rows(path: Path) -> int:
    with path.open('r', encoding='utf-8', newline='') as handle:
        return max(sum(1 for _ in handle) - 1, 0)


def file_size(path: Path) -> int:
    return path.stat().st_size if path.exists() else 0


def clean_unselected_data() -> None:
    keep = {dataset['slug'] for dataset in SELECTED_DOTSTAT_DATASETS}
    keep.update({'manifest.csv', 'geo'})
    if not RAW_DATA_DIR.exists():
        return
    for path in RAW_DATA_DIR.iterdir():
        if path.name in keep:
            continue
        if path.is_dir():
            print(f'remove stale dataset {path.relative_to(PROJECT_ROOT)}')
            shutil.rmtree(path)
        elif path.is_file():
            print(f'remove stale file {path.relative_to(PROJECT_ROOT)}')
            path.unlink()

## **1.3. Run data collection**

Each dataset is saved with its source metadata. The resulting manifest provides a compact audit of the complete collection and is the input to the exploration and processing notebooks.

In [ ]:
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

if CLEAN_UNSELECTED_DATASETS:
    clean_unselected_data()

downloaded_at = datetime.now(timezone.utc).replace(microsecond=0).isoformat()
manifest_rows = []

for dataset in SELECTED_DOTSTAT_DATASETS:
    dataset_dir = RAW_DATA_DIR / dataset['slug']
    dataset_dir.mkdir(parents=True, exist_ok=True)

    api_url = dotstat_url(dataset)
    data_file = dataset_dir / 'data.csv'
    download_file(api_url, data_file, force=FORCE_DOWNLOAD)

    metadata = {
        'api_url': api_url,
        'dataflow': dataset['dataflow'],
        'downloaded_at_utc': downloaded_at,
        'files': [{'path': 'data.csv', 'size_bytes': file_size(data_file)}],
        'front': dataset['front'],
        'key': dataset['key'],
        'source_url': dataset['source_url'],
        'title': dataset['title'],
        'type': 'dotstat_csv',
    }
    write_json(dataset_dir / 'metadata.json', metadata)

    manifest_rows.append({
        'slug': dataset['slug'],
        'title': dataset['title'],
        'front': dataset['front'],
        'type': 'dotstat_csv',
        'local_files': 'data.csv; metadata.json',
        'size_bytes': file_size(data_file),
        'rows': count_csv_rows(data_file),
        'source_url': dataset['source_url'],
        'api_url': api_url,
    })

manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(RAW_DATA_DIR / 'manifest.csv', index=False)

display(manifest[['slug', 'front', 'title', 'rows', 'size_bytes']])

## **1.4. Source-data inspection**

The eight downloaded tables are loaded independently for an immediate schema and content check. No transformations are applied at this stage.

In [ ]:
MEAN_SURFACE_TEMPERATURE_ANOMALIES = pd.read_csv(RAW_DATA_DIR / 'mean_surface_temperature_anomalies' / 'data.csv')
display(Markdown('### **Mean surface temperature anomalies**'))
display(MEAN_SURFACE_TEMPERATURE_ANOMALIES.head())

In [ ]:
MEAN_SEA_SURFACE_TEMPERATURE_ANOMALIES = pd.read_csv(RAW_DATA_DIR / 'mean_sea_surface_temperature_anomalies' / 'data.csv')
display(Markdown('### **Mean sea surface temperature anomalies**'))
display(MEAN_SEA_SURFACE_TEMPERATURE_ANOMALIES.head())

In [ ]:
RAINFALL_ANOMALIES = pd.read_csv(RAW_DATA_DIR / 'rainfall_anomalies' / 'data.csv')
display(Markdown('### **Rainfall anomalies**'))
display(RAINFALL_ANOMALIES.head())

In [ ]:
DISASTER_DIRECTLY_AFFECTED_PERSONS = pd.read_csv(RAW_DATA_DIR / 'disaster_directly_affected_persons' / 'data.csv')
display(Markdown('### **Disaster directly affected persons**'))
display(DISASTER_DIRECTLY_AFFECTED_PERSONS.head())

In [ ]:
DIRECT_DISASTER_ECONOMIC_LOSS = pd.read_csv(RAW_DATA_DIR / 'direct_disaster_economic_loss' / 'data.csv')
display(Markdown('### **Direct disaster economic loss**'))
display(DIRECT_DISASTER_ECONOMIC_LOSS.head())

In [ ]:
CROP_YIELD = pd.read_csv(RAW_DATA_DIR / 'crop_yield' / 'data.csv')
display(Markdown('### **Crop yield**'))
display(CROP_YIELD.head())

In [ ]:
LIVESTOCK_YIELD = pd.read_csv(RAW_DATA_DIR / 'livestock_yield' / 'data.csv')
display(Markdown('### **Livestock yield**'))
display(LIVESTOCK_YIELD.head())

In [ ]:
AGRICULTURAL_YIELD_DISAGGREGATED = pd.read_csv(RAW_DATA_DIR / 'agricultural_yield_disaggregated' / 'data.csv')
display(Markdown('### **Agricultural yield - disaggregated**'))
display(AGRICULTURAL_YIELD_DISAGGREGATED.head())